---
title: "Від чого залежить ціна діаманта: регресійне моделювання"
subtitle: "Лабораторна робота №3. Основи регресійного моделювання"
author: "Демчук Владислав"
date: last-modified
lang: uk
format:
  html:
    toc: true
    toc-depth: 2
    number-sections: true
    code-fold: true
    code-summary: "Код"
    embed-resources: true
    df-print: kable
    fig-dpi: 120
bibliography: references.bib
nocite: |
  @james2023, @nist, @kuhn2022, @sandve2013
execute:
  warning: false
  message: false
jupyter: python3
---

In [ ]:
#| label: setup
#| code-summary: "Пакети й налаштування"
import os
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

numbers = {}


def usd(x):
    return f"{x:,.0f}".replace(",", " ") + " $"

# Вступ

## Предметна область, ціль і питання

Ціна діаманта на роздрібному ринку визначається так званими «4C»: вагою (*carat*), огранкою (*cut*),
кольором (*color*) і чистотою (*clarity*). Продовжую лабораторну №2 на тому самому наборі **Diamonds**.

- **Одиниця спостереження** — один огранений круглий діамант з каталогу онлайн-продавця.
- **Ціль** — `price`, ціна в доларах США (ціни 2008 року), числова й неперервна.
- **Дослідницьке питання.** Наскільки точно чотири характеристики з сертифіката (4C) прогнозують ціну
  *нового* діаманта з того самого ринку і як ціна пов'язана з вагою, якщо огранка, колір і чистота однакові?
- **Для яких спостережень потрібен прогноз.** Для круглих діамантів того самого типу й періоду,
  у межах діапазону даних (0.2–5 ct, ціна до 18 823 $). Наприклад, покупець чи продавець хоче оцінити,
  чи справедлива ціна каменя з відомими 4C.

## Що підказує література

1. **Chu [-@chu2001]** будує модель ціни діамантів за 4C на даних сінгапурських оголошень. Ціна зростає
   з вагою швидше, ніж пропорційно, а якісні характеристики працюють як надбавки до ціни.
   Звідси **вибір змінних** (4C як основні предиктори) і **форма моделі**: логарифм ціни дає
   мультиплікативні (відсоткові) ефекти колір/чистота і рівномірніший розкид залишків.
2. **Scott і Yelowitz [-@scott2010]** на даних трьох великих онлайн-продавців показали **стрибки ціни на
   «круглих» вагах**: камінь 0.5 ct дорожчий за 0.49 ct на 18 % і більше, а 1 ct дорожчий за трохи легший
   на 5–10 %. Це підказує **межі інтерпретації**: зв'язок ціни з вагою не гладкий, тому проста
   степенева модель систематично помилятиметься біля порогів 0.5, 1, 1.5, 2 ct. Звідси моя модифікація
   регресії (порогові змінні і сплайн).
3. **Wickham і Grolemund [-@r4ds]** на цьому самому наборі показують, що в шкалі log–log зв'язок ціни з
   вагою майже лінійний і що «гірша» якість здається дорожчою через змішування з вагою. Це методологічна
   підстава завжди оцінювати ефекти якості **за сталої ваги**.

## Гіпотези (записані до оцінювання моделей)

- **H1 (вага, з EDA і літератури).** За однакових огранки, кольору й чистоти ціна зростає з вагою
  **швидше, ніж пропорційно**: еластичність ціни за вагою $\beta_{\log carat} > 1$.
  *Перевірка:* знак, величина й 95 % довірчий інтервал коефіцієнта при $\log(carat)$ у заздалегідь
  описаній моделі M2 (нижче), стійкі до гетероскедастичності стандартні похибки HC3.
- **H2 (чистота, з EDA і літератури).** За однакової ваги, огранки й кольору ціна **монотонно зростає**
  з чистотою від I1 до IF, і надбавка IF порівняно з I1 перевищує 100 %.
  *Перевірка:* коефіцієнти рівнів `clarity` у M2: знаки, порядок і 95 % ДІ надбавки $e^{\beta}-1$.

::: {.callout-warning}
Обидві гіпотези **виникли з EDA** лабораторної №2 (нахил 1.68 у log–log; чистота найсильніше
пов'язана з ціною каменів 1 ct), яке охоплювало **весь** набір. Тому їх перевірка на тих самих даних
є **розвідувальною**, а не підтверджувальною.
:::

# Методи

## Дані

In [ ]:
#| label: data
raw_path = "data/raw/diamonds.csv"
url = "https://raw.githubusercontent.com/tidyverse/ggplot2/main/data-raw/diamonds.csv"

if not os.path.exists(raw_path):
    os.makedirs("data/raw", exist_ok=True)
    pd.read_csv(url).to_csv(raw_path, index=False)

with open(raw_path, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()

raw = pd.read_csv(raw_path)

df = raw.drop_duplicates()
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]
df["depth_check"] = df["z"] / ((df["x"] + df["y"]) / 2) * 100
df = df[abs(df["depth_check"] - df["depth"]) <= 5]
df = df.drop(columns="depth_check")
df = df.reset_index(drop=True)

df["cut"] = pd.Categorical(df["cut"], categories=["Fair", "Good", "Very Good", "Premium", "Ideal"], ordered=True)
df["color"] = pd.Categorical(df["color"], categories=["J", "I", "H", "G", "F", "E", "D"], ordered=True)
df["clarity"] = pd.Categorical(df["clarity"], categories=["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"], ordered=True)

df["cut_code"] = df["cut"].cat.codes
df["color_code"] = df["color"].cat.codes
df["clarity_code"] = df["clarity"].cat.codes

df["log_price"] = np.log(df["price"])
df["log_carat"] = np.log(df["carat"])

numbers["n_raw"] = len(raw)
numbers["n_clean"] = len(df)
numbers["sha"] = sha

print("Рядків у файлі:", len(raw))
print("Рядків після очищення:", len(df))
print("SHA-256 файлу:", sha)

- **Джерело:** набір `diamonds` з пакета ggplot2 [@ggplot2], файл `data-raw/diamonds.csv` з репозиторію
  tidyverse/ggplot2 (гілка `main`), отримано **листопад 2026**. Контрольна сума SHA-256 файлу
  `{python} sha[:16]`… записана в звіті, тож ту саму версію можна перевірити.
- **Умови використання:** ліцензія MIT пакета ggplot2; дані відкриті.
- **Розмір:** `{python} f"{len(raw):,}".replace(",", " ")` рядків і 10 змінних; після очищення
  `{python} f"{len(df):,}".replace(",", " ")` (вилучено повні дублікати, камені з нульовими розмірами і
  рядки, де `depth` суперечить розмірам x, y, z).
- **Обмеження даних:** спостережні дані одного ринку 2008 року; **немає дати й продавця**; ціни обрізані
  зверху (максимум 18 823 $); у наборі тільки круглі камені. Модель не можна переносити на інші роки,
  форми чи синтетичні діаманти.

## Поділ і захист від витоку

Спостереження незалежні (один рядок — окремий камінь, повторних вимірювань і часу немає), тому
**випадковий поділ 80/20 з `seed = 42`**. Поділ я зафіксував **до** підбору моделей; тестову частину
не використовую аж до фінальної оцінки.

In [ ]:
#| label: split
train, test = train_test_split(df, test_size=0.2, random_state=42)

print("Навчальна частина:", len(train), "рядків, середня ціна", round(train["price"].mean()), "$")
print("Тестова частина:", len(test), "рядків, середня ціна", round(test["price"].mean()), "$")

- Усі графіки розділу 2.3 і всю діагностику я побудував **тільки на навчальній частині**.
- **Обмеження незалежності тесту.** Лабораторна №2 бачила весь набір, зокрема майбутню тестову частину.
  Вона вже підказала рішення: логарифмувати ціну й вагу, відкинути x, y, z (дублюють вагу), не брати
  `depth` і `table` (слабкий зв'язок), а також гіпотези H1–H2. Тому тестова оцінка трохи оптимістична.
  Нових даних з того самого ринку немає, отже повністю незалежної перевірки зробити не можу.
- **Доступність предикторів.** 4C вказані в сертифікаті до продажу, тож вони відомі на момент прогнозу.
  Ціну я ніде не використовую як предиктор.
- **Перехресна валідація:** 5-кратна `KFold(shuffle=True, random_state=42)` на навчальній частині,
  **однакові розбиття для всіх моделей**. Кожну модель на кожному розбитті я навчаю тільки на його
  навчальній частині, а помилку рахую на валідаційній. Лінійні моделі задані формулами `statsmodels`:
  кодування категорій і вузли сплайна визначаються лише за навчальною частиною розбиття, а логарифми й
  пороги ваги рахуються для кожного рядка окремо і нічого не беруть з інших даних.
  Пропусків немає, тож імпутація не потрібна; масштабування лінійній регресії без штрафу не потрібне.

## Що з EDA переношу в моделювання

In [ ]:
#| label: fig-eda
#| fig-cap: "EDA на навчальній частині. Ліворуч: ціна має сильну праву асиметрію, після логарифма розподіл майже симетричний (кілька піків — популярні ваги каменів). Праворуч: у шкалі log–log ціна майже лінійно залежить від ваги, але видно вертикальні «сходинки» на 0.5, 1, 1.5 і 2 ct."
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

sns.histplot(train["price"], bins=60, ax=ax[0], color="steelblue")
ax[0].set_title("Ціна")
ax[0].set_xlabel("Ціна, $")
ax[0].set_ylabel("Кількість діамантів")

sns.histplot(train["log_price"], bins=60, ax=ax[1], color="seagreen")
ax[1].set_title("Логарифм ціни")
ax[1].set_xlabel("ln(ціна, $)")
ax[1].set_ylabel("")

ax[2].scatter(train["carat"], train["price"], s=2, alpha=0.08)
ax[2].set_xscale("log")
ax[2].set_yscale("log")
ax[2].axvline(0.5, color="red", lw=0.8, ls="--")
ax[2].axvline(1, color="red", lw=0.8, ls="--")
ax[2].axvline(1.5, color="red", lw=0.8, ls="--")
ax[2].axvline(2, color="red", lw=0.8, ls="--")
ax[2].minorticks_off()
ax[2].set_xticks([0.2, 0.3, 0.5, 1, 2, 3, 5], ["0.2", "0.3", "0.5", "1", "2", "3", "5"])
ax[2].set_yticks([500, 1000, 2000, 5000, 10000], ["500", "1000", "2000", "5000", "10000"])
ax[2].set_title("Ціна і вага (log–log)")
ax[2].set_xlabel("Вага, ct (log)")
ax[2].set_ylabel("Ціна, $ (log)")

plt.tight_layout()
plt.savefig("figures/01_eda_target.png", dpi=150, bbox_inches="tight")
plt.show()

numbers["skew_price"] = round(float(train["price"].skew()), 2)
numbers["skew_log"] = round(float(train["log_price"].skew()), 2)

In [ ]:
#| label: fig-eda-clarity
#| fig-cap: "Медіанна ціна за чистотою: без контролю ваги (ліворуч) найчистіші камені здаються дешевшими; серед каменів 0.9–1.1 ct (праворуч) ціна чітко зростає з чистотою. Отже, ефекти якості треба оцінювати за сталої ваги."
median_all = train.groupby("clarity", observed=True)["price"].median()

one_carat = train[(train["carat"] >= 0.9) & (train["carat"] <= 1.1)]
median_one_carat = one_carat.groupby("clarity", observed=True)["price"].median()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].bar(median_all.index.astype(str), median_all.values, color="lightgray")
ax[0].set_title("Усі камені")
ax[0].set_xlabel("Чистота (від гіршої до кращої)")
ax[0].set_ylabel("Медіанна ціна, $")

ax[1].bar(median_one_carat.index.astype(str), median_one_carat.values, color="steelblue")
ax[1].set_title("Камені 0.9–1.1 ct")
ax[1].set_xlabel("Чистота (від гіршої до кращої)")

plt.tight_layout()
plt.savefig("figures/02_eda_clarity.png", dpi=150, bbox_inches="tight")
plt.show()

Найважливіші висновки з лабораторної №2, перевірені тут на навчальній частині:

- **Розподіл цілі:** асиметрія ціни `{python} numbers["skew_price"]`, логарифма ціни `{python} numbers["skew_log"]`.
  *Рішення:* модель для $\ln(price)$; ефекти тоді у відсотках, а розкид залишків рівніший.
- **Пропуски:** немає. **Викиди:** дорогі й великі камені реальні, їх не видаляю; помилкові рядки
  вже вилучено на очищенні.
- **Зв'язки й нелінійність:** зв'язок ціни з вагою степеневий (у log–log майже пряма) зі «сходинками»
  на круглих вагах. *Рішення:* $\ln(carat)$ у базовій лінійній моделі; сплайн і пороги в модифікації.
- **Змішування:** «краща» якість частіше в дрібних каменях (@fig-eda-clarity). *Рішення:* завжди
  тримаю вагу в моделі, коли інтерпретую огранку, колір і чистоту.
- **Дублювання:** x, y, z майже ідеально корелюють з вагою (r ≈ 0.98), тому не входять у модель.

## Метрики й правило вибору (зафіксовано до відкриття тесту)

- **Основна метрика — MAE**, у доларах: «на скільки в середньому помиляється оцінка ціни». Розподіл ціни
  дуже асиметричний, і RMSE домінували б кілька найдорожчих каменів. RMSE і $R^2$ подаю як додаткові.
- **Правило вибору:** модель з найменшим середнім MAE на 5 розбиттях. Якщо простіша модель гірша менше
  ніж на одне стандартне відхилення MAE за розбиттями, беру простішу.
- На тесті один раз оцінюю обрану модель і константний орієнтир. Найкращу **лінійну** модель (вона
  потрібна для інтерпретації гіпотез) теж один раз оцінюю на тесті, це рішення прийняте заздалегідь.
- Для моделей на $\ln(price)$ прогноз у доларах — $e^{\hat y}$, тобто оцінка **медіани** ціни.
  Вона узгоджується з MAE, але трохи занижує середню ціну [@duan1983].

# Результати

In [ ]:
#| label: models
#| code-summary: "Формули всіх моделей"
formula_m1 = "price ~ carat + C(cut) + C(color) + C(clarity)"

formula_m2 = "log_price ~ log_carat + C(cut) + C(color) + C(clarity)"

formula_m3 = ("log_price ~ bs(log_carat, df=6, lower_bound=-2.5, upper_bound=2)"
              " + I(carat >= 0.5) + I(carat >= 1) + I(carat >= 1.5) + I(carat >= 2)"
              " + C(cut) + C(color) + C(clarity)")

formula_m4 = ("log_price ~ bs(log_carat, df=6, lower_bound=-2.5, upper_bound=2)"
              " + I(carat >= 0.5) + I(carat >= 1) + I(carat >= 1.5) + I(carat >= 2)"
              " + C(cut) + C(color) * C(clarity)")

boost_columns = ["carat", "cut_code", "color_code", "clarity_code"]

model_names = {
    "M0": "M0 константа (середнє)",
    "M1": "M1 лінійна, ціна в $",
    "M2": "M2 лінійна log–log",
    "M3": "M3 log–log + сплайн і пороги ваги",
    "M4": "M4 = M3 + взаємодія колір×чистота",
    "M5": "M5 градієнтний бустинг (додатково)",
}

## Орієнтир: константний прогноз

M0 завжди прогнозує середню ціну навчальної частини, `{python} usd(train["price"].mean())`. Це точність
«без жодних предикторів»: будь-яка корисна модель має бути помітно кращою (результати в @tbl-cv).

## Базова регресія

Спершу найпростіша лінійна модель у доларах (M1):
$price = \beta_0 + \beta_1\,carat + \text{cut} + \text{color} + \text{clarity} + \varepsilon$.

In [ ]:
#| label: fig-m1-resid
#| fig-cap: "M1 (ціна в $): залишки проти прогнозів. Видно дугу (нелінійність), «лійку» (розкид зростає з ціною) і від'ємні прогнози для дрібних каменів."
m1 = smf.ols(formula_m1, data=train).fit()

plt.figure(figsize=(7.5, 4))
plt.scatter(m1.fittedvalues, m1.resid, s=2, alpha=0.1)
plt.axhline(0, color="red")
plt.title("M1: залишки проти прогнозів")
plt.xlabel("Прогноз, $")
plt.ylabel("Залишок, $")
plt.tight_layout()
plt.savefig("figures/03_m1_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

numbers["m1_neg"] = int((m1.fittedvalues < 0).sum())

M1 має три вади: `{python} numbers["m1_neg"]` каменів отримують **від'ємну** ціну, залишки утворюють дугу
(зв'язок не лінійний), а розкид росте з ціною. Тому як **базову регресію для висновків** беру
лог-лінійну модель, яку підказали EDA й література (M2):

$$\ln(price) = \beta_0 + \beta_1 \ln(carat) + \sum \gamma_{cut} + \sum \delta_{color} + \sum \theta_{clarity} + \varepsilon$$

Еталонні рівні: огранка Fair, колір J, чистота I1. Коефіцієнти я оцінив на всій навчальній частині
OLS зі стійкими до гетероскедастичності похибками HC3.

In [ ]:
#| label: tbl-m2
#| tbl-cap: "M2: ключові коефіцієнти, 95 % ДІ (HC3) і надбавка до ціни за сталих інших змінних"
m2 = smf.ols(formula_m2, data=train).fit(cov_type="HC3")
ci = m2.conf_int()

coef = pd.DataFrame()
coef["β"] = m2.params
coef["ДІ 2.5 %"] = ci[0]
coef["ДІ 97.5 %"] = ci[1]
coef["надбавка, %"] = (np.exp(m2.params) - 1) * 100
coef.loc["Intercept", "надбавка, %"] = np.nan
coef.loc["log_carat", "надбавка, %"] = np.nan
coef = coef.round(3)
coef.to_csv("results/m2_coefficients.csv")

b1 = float(m2.params["log_carat"])
clarity_beta = m2.params[["C(clarity)[T.SI2]", "C(clarity)[T.SI1]", "C(clarity)[T.VS2]",
                          "C(clarity)[T.VS1]", "C(clarity)[T.VVS2]", "C(clarity)[T.VVS1]",
                          "C(clarity)[T.IF]"]]

numbers["b1"] = round(b1, 3)
numbers["b1_lo"] = round(float(ci.loc["log_carat", 0]), 3)
numbers["b1_hi"] = round(float(ci.loc["log_carat", 1]), 3)
numbers["r2_m2"] = round(float(m2.rsquared), 4)
numbers["si2_prem"] = round(float(np.exp(m2.params["C(clarity)[T.SI2]"]) - 1) * 100)
numbers["if_prem"] = round(float(np.exp(m2.params["C(clarity)[T.IF]"]) - 1) * 100)
numbers["if_lo"] = round(float(np.exp(ci.loc["C(clarity)[T.IF]", 0]) - 1) * 100)
numbers["if_hi"] = round(float(np.exp(ci.loc["C(clarity)[T.IF]", 1]) - 1) * 100)
numbers["clar_monotone"] = bool(clarity_beta.is_monotonic_increasing and (clarity_beta > 0).all())
numbers["d_prem"] = round(float(np.exp(m2.params["C(color)[T.D]"]) - 1) * 100)
numbers["ideal_prem"] = round(float(np.exp(m2.params["C(cut)[T.Ideal]"]) - 1) * 100)

coef

**Інтерпретація в одиницях задачі** ($R^2$ на шкалі логарифма = `{python} numbers["r2_m2"]`):

- **Вага:** $\hat\beta_1 =$ `{python} numbers["b1"]` (95 % ДІ `{python} numbers["b1_lo"]`–`{python} numbers["b1_hi"]`).
  Камінь на 1 % важчий коштує в середньому на ≈ `{python} round(b1, 2)` % дорожче. Камінь, удвічі
  важчий за ту саму якість, коштує в $2^{\hat\beta_1} \approx$ `{python} round(2 ** b1, 1)` раза більше.
- **Чистота:** порівняно з I1 надбавка зростає від `{python} numbers["si2_prem"]` % (SI2) до
  `{python} numbers["if_prem"]` % (IF, 95 % ДІ `{python} numbers["if_lo"]`–`{python} numbers["if_hi"]` %).
- **Колір:** найкращий колір D дорожчий за J на ≈ `{python} numbers["d_prem"]` %.
- **Огранка:** Ideal дорожча за Fair лише на ≈ `{python} numbers["ideal_prem"]` %. Огранка важить найменше з 4C.

### Діагностика залишків M2

In [ ]:
#| label: fig-m2-diag
#| fig-cap: "Діагностика M2. (а) Залишки проти прогнозів: дуги й лійки M1 майже немає. (б) Середній залишок за вагою: систематична хвиля від −0.12 до +0.11 зі зламами біля 0.5, 1, 1.5 і 2 ct. (в) Q–Q графік: правий хвіст важчий за нормальний."
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

ax[0].scatter(m2.fittedvalues, m2.resid, s=2, alpha=0.08)
ax[0].axhline(0, color="red")
ax[0].set_title("(а) Залишки проти прогнозів")
ax[0].set_xlabel("Прогноз ln(ціна)")
ax[0].set_ylabel("Залишок")

bins = np.arange(0.2, 2.31, 0.05)
train_resid = train.copy()
train_resid["resid"] = m2.resid
train_resid["weight_bin"] = pd.cut(train_resid["carat"], bins)
resid_by_weight = train_resid.groupby("weight_bin", observed=True)["resid"].agg(["mean", "size"])
resid_by_weight = resid_by_weight[resid_by_weight["size"] >= 30]
bin_middle = []
for interval in resid_by_weight.index:
    bin_middle.append(interval.mid)

ax[1].plot(bin_middle, resid_by_weight["mean"], marker="o", ms=3)
ax[1].axhline(0, color="red")
ax[1].axvline(0.5, color="gray", ls="--", lw=0.8)
ax[1].axvline(1, color="gray", ls="--", lw=0.8)
ax[1].axvline(1.5, color="gray", ls="--", lw=0.8)
ax[1].axvline(2, color="gray", ls="--", lw=0.8)
ax[1].set_title("(б) Середній залишок за вагою")
ax[1].set_xlabel("Вага, ct")
ax[1].set_ylabel("Середній залишок (ln)")

stats.probplot(m2.resid.sample(4000, random_state=42), dist="norm", plot=ax[2])
ax[2].set_title("(в) Q–Q графік залишків (4000 точок)")

plt.tight_layout()
plt.savefig("figures/04_m2_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#| label: influence
m2_simple = smf.ols(formula_m2, data=train).fit()

bp_m1 = het_breuschpagan(m1.resid, m1.model.exog)[1]
bp_m2 = het_breuschpagan(m2_simple.resid, m2_simple.model.exog)[1]

cook = OLSInfluence(m2_simple).cooks_distance[0]
n_cook = int((cook > 4 / len(train)).sum())

n_top = int(0.01 * len(train))
top_rows = np.argsort(cook)[-n_top:]
train_without_top = train.drop(train.index[top_rows])
m2_without_top = smf.ols(formula_m2, data=train_without_top).fit()

spread_expensive = m1.resid[m1.fittedvalues > 10000].std()
spread_cheap = m1.resid[m1.fittedvalues < 2000].std()

numbers["n_cook"] = n_cook
numbers["cook_max"] = round(float(cook.max()), 4)
numbers["b1_wo"] = round(float(m2_without_top.params["log_carat"]), 3)
numbers["sd_ratio"] = round(float(spread_expensive / spread_cheap), 1)

print("Breusch–Pagan, p (M1):", bp_m1)
print("Breusch–Pagan, p (M2):", bp_m2)
print("Точок з D Кука > 4/n:", n_cook)
print("Максимальна D Кука:", numbers["cook_max"])
print("β ln(carat) без 1 % найвпливовіших точок:", numbers["b1_wo"])

**Що показала діагностика і що з цього випливає:**

- **Нелінійність.** Логарифми прибрали велику дугу M1, але лишилась систематична «хвиля» за вагою
  (@fig-m2-diag б). Камені 0.4–0.5 ct і 0.55–0.85 ct модель переоцінює (середній залишок до −0.12,
  тобто прогноз на ≈ 11 % завищений), а камені близько 1 ct і найдрібніші недооцінює (до +0.11). Злами
  кривої припадають на 0.5, 1, 1.5 і 2 ct, тобто на «круглі» ваги [@scott2010]. *Наслідок для прогнозу:*
  систематична похибка до ≈ 10 % для цілих груп каменів. *Наслідок для коефіцієнтів:* $\beta_1$ є усередненою
  еластичністю, а не сталою для всіх ваг.
- **Неоднаковий розкид.** У M1 розкид залишків для дорогих каменів у `{python} numbers["sd_ratio"]` раза
  більший, ніж для дешевих. У M2 він значно рівніший, проте тест Бройша–Пагана все одно відхиляє
  гомоскедастичність (при n ≈ 43 000 він чутливий навіть до малих відхилень). Тому для ДІ я використав
  **HC3**. Q–Q графік показує важкі хвости, тож прогнозні інтервали на основі нормальності були б
  завузькими.
- **Впливові спостереження.** `{python} numbers["n_cook"]` точок мають D Кука > 4/n, але максимум лише
  `{python} numbers["cook_max"]` (далеко від 1). Без 1 % найвпливовіших точок
  $\hat\beta_1$ = `{python} numbers["b1_wo"]` замість `{python} numbers["b1"]`. Висновки від них не залежать.

## Спроба покращення

**Проблема:** хвиля в залишках за вагою (@fig-m2-diag б). **Модифікації:**

- **M3** замінює пряму $\ln(carat)$ на кубічний сплайн від $\ln(carat)$ (6 ступенів свободи, вузли
  за навчальною частиною) і додає 4 порогові змінні $[carat \ge 0.5], [\ge 1], [\ge 1.5], [\ge 2]$
  з роботи Scott і Yelowitz.
- **M4** додатково дозволяє надбавці за колір залежати від чистоти (взаємодія color × clarity).
  У лабораторній №2 теплова карта медіанних цін «колір × чистота» для каменів 1 ct показала, що ці
  надбавки не просто додаються одна до одної.

Ridge не розглядаю: предикторів лише ~20 на ~43 000 спостережень, мультиколінеарності між 4C немає,
тож штраф майже не змінить коефіцієнтів.

## Додаткова модель

**M5** — градієнтний бустинг (`HistGradientBoostingRegressor`, параметри за замовчуванням) на тих самих
4 предикторах і тій самій цілі $\ln(price)$. Він сам знаходить пороги й взаємодії, але не має
коефіцієнтів для інтерпретації.

## Порівняння за перехресною валідацією

In [ ]:
#| label: tbl-cv
#| tbl-cap: "5-кратна перехресна валідація на навчальній частині (однакові розбиття). Метрики в доларах; середнє ± стандартне відхилення за 5 розбиттями."
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae = {"M0": [], "M1": [], "M2": [], "M3": [], "M4": [], "M5": []}
rmse = {"M0": [], "M1": [], "M2": [], "M3": [], "M4": [], "M5": []}
r2 = {"M0": [], "M1": [], "M2": [], "M3": [], "M4": [], "M5": []}

for train_index, valid_index in kfold.split(train):
    part_train = train.iloc[train_index]
    part_valid = train.iloc[valid_index]
    predictions = {}

    predictions["M0"] = np.full(len(part_valid), part_train["price"].mean())

    model = smf.ols(formula_m1, data=part_train).fit()
    predictions["M1"] = model.predict(part_valid)

    model = smf.ols(formula_m2, data=part_train).fit()
    predictions["M2"] = np.exp(model.predict(part_valid))

    model = smf.ols(formula_m3, data=part_train).fit()
    predictions["M3"] = np.exp(model.predict(part_valid))

    model = smf.ols(formula_m4, data=part_train).fit()
    predictions["M4"] = np.exp(model.predict(part_valid))

    booster = HistGradientBoostingRegressor(random_state=42)
    booster.fit(part_train[boost_columns], part_train["log_price"])
    predictions["M5"] = np.exp(booster.predict(part_valid[boost_columns]))

    for name in predictions:
        mae[name].append(mean_absolute_error(part_valid["price"], predictions[name]))
        rmse[name].append(root_mean_squared_error(part_valid["price"], predictions[name]))
        r2[name].append(r2_score(part_valid["price"], predictions[name]))

rows = []
for name in model_names:
    rows.append({
        "код": name,
        "модель": model_names[name],
        "MAE, $": np.mean(mae[name]),
        "± MAE": np.std(mae[name]),
        "RMSE, $": np.mean(rmse[name]),
        "± RMSE": np.std(rmse[name]),
        "R²": np.mean(r2[name]),
    })

cv_table = pd.DataFrame(rows).set_index("код")

cv_rounded = cv_table.round({"MAE, $": 0, "± MAE": 0, "RMSE, $": 0, "± RMSE": 0, "R²": 3})
cv_rounded.to_csv("results/cv_comparison.csv", index=False)
cv_rounded

In [ ]:
#| label: choice
cv_sorted = cv_table.sort_values("MAE, $")
best = cv_sorted.index[0]
second = cv_sorted.index[1]
difference = cv_table.loc[second, "MAE, $"] - cv_table.loc[best, "MAE, $"]

if second < best and difference < cv_table.loc[best, "± MAE"]:
    chosen = second
else:
    chosen = best

linear_models = cv_table.loc[["M1", "M2", "M3", "M4"]]
best_linear = linear_models["MAE, $"].idxmin()

numbers["chosen"] = model_names[chosen]
numbers["best_linear"] = model_names[best_linear]
numbers["gain_m3"] = round(float(1 - cv_table.loc["M3", "MAE, $"] / cv_table.loc["M2", "MAE, $"]) * 100, 1)
numbers["gain_m4"] = round(float(1 - cv_table.loc["M4", "MAE, $"] / cv_table.loc["M3", "MAE, $"]) * 100, 1)
numbers["gain_gb"] = round(float(1 - cv_table.loc["M5", "MAE, $"] / cv_table.loc[best_linear, "MAE, $"]) * 100, 1)

print("Обрана модель (до відкриття тесту):", model_names[chosen])
print("Найкраща лінійна модель:", model_names[best_linear])

**Що показала валідація:**

- Усі регресії значно кращі за орієнтир M0. Перехід до логарифмів (M1 → M2) зменшує MAE приблизно
  вдвічі. Отже, форма зв'язку важить більше за вибір предикторів.
- **Сплайн і пороги (M3)** зменшують MAE ще на `{python} numbers["gain_m3"]` % порівняно з M2, різниця
  набагато більша за розкид між розбиттями. Модифікація розв'язала проблему, яку показала діагностика.
- **Взаємодія колір × чистота (M4)** зменшує MAE ще на `{python} numbers["gain_m4"]` % порівняно з M3.
- **Бустинг (M5)** точніший за найкращу лінійну модель на `{python} numbers["gain_gb"]` %, але не має
  коефіцієнтів. Тому гіпотези перевіряю в M2, а для прогнозу беру модель за правилом вибору.

## Фінальна оцінка на тесті

In [ ]:
#| label: tbl-test
#| tbl-cap: "Одноразова оцінка на тестовій частині. Моделі перенавчено на всій навчальній частині."
pred_m0 = np.full(len(test), train["price"].mean())

final_booster = HistGradientBoostingRegressor(random_state=42)
final_booster.fit(train[boost_columns], train["log_price"])
pred_m5 = np.exp(final_booster.predict(test[boost_columns]))

final_m4 = smf.ols(formula_m4, data=train).fit()
pred_m4 = np.exp(final_m4.predict(test))

test_rows = []
for name, prediction in [("M0", pred_m0), ("M5", pred_m5), ("M4", pred_m4)]:
    relative_error = abs(test["price"] - prediction) / test["price"] * 100
    test_rows.append({
        "модель": model_names[name],
        "MAE, $": round(float(mean_absolute_error(test["price"], prediction))),
        "RMSE, $": round(float(root_mean_squared_error(test["price"], prediction))),
        "R²": round(float(r2_score(test["price"], prediction)), 3),
        "медіана |помилки| / ціна, %": round(float(np.median(relative_error)), 1),
    })

test_table = pd.DataFrame(test_rows)
test_table.to_csv("results/test_metrics.csv", index=False)
numbers["test"] = test_rows
test_table

In [ ]:
#| label: fig-test
#| fig-cap: "Тест: фактична ціна проти прогнозу обраної моделі (log-шкали; червона лінія — ідеальний прогноз) і абсолютна помилка за групами ваги."
err = test.copy()
err["pred"] = pred_m5
err["abs_err"] = abs(err["price"] - err["pred"])
err["rel_err"] = err["abs_err"] / err["price"] * 100
err["група ваги"] = pd.cut(err["carat"], [0, 0.5, 0.75, 1, 1.5, 2, 5.1],
                           labels=["до 0.5", "0.5–0.75", "0.75–1", "1–1.5", "1.5–2", "понад 2"])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

ax[0].scatter(err["price"], err["pred"], s=3, alpha=0.15)
ax[0].plot([300, 20000], [300, 20000], color="red")
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlim(300, 20000)
ax[0].set_ylim(300, 20000)
ax[0].set_title("Фактичне і прогноз: M5")
ax[0].set_xlabel("Фактична ціна, $")
ax[0].set_ylabel("Прогноз, $")

sns.boxplot(data=err, x="група ваги", y="abs_err", ax=ax[1], showfliers=False, color="lightblue")
ax[1].set_title("Абсолютна помилка за вагою")
ax[1].set_xlabel("Вага, ct")
ax[1].set_ylabel("|помилка|, $")

plt.tight_layout()
plt.savefig("figures/05_test_pred.png", dpi=150, bbox_inches="tight")
plt.show()

by_weight = err.groupby("група ваги", observed=True).agg(
    n=("price", "size"),
    median_price=("price", "median"),
    MAE=("abs_err", "mean"),
    median_rel_err=("rel_err", "median"),
)
by_weight = by_weight.round(1)
by_weight.to_csv("results/test_errors_by_carat.csv")

small = by_weight.iloc[0]
large = by_weight.iloc[-1]
numbers["rel_err_small"] = float(small["median_rel_err"])
numbers["rel_err_large"] = float(large["median_rel_err"])

by_weight

In [ ]:
#| label: worst
expensive = err[err["price"] > 15000]
big_stones = err[err["carat"] >= 1.5]

numbers["n_cap"] = len(expensive)
numbers["mae_cap"] = round(float(expensive["abs_err"].mean()))
numbers["share_big_n"] = round(len(big_stones) / len(err) * 100)
numbers["share_big_err"] = round(float(big_stones["abs_err"].sum() / err["abs_err"].sum()) * 100)

worst = err.sort_values("abs_err", ascending=False).head(6)
worst[["carat", "cut", "color", "clarity", "price", "pred", "abs_err"]].round(0)

**Де помилки найбільші.** У доларах помилка росте з вагою й ціною: камені від 1.5 ct становлять
`{python} numbers["share_big_n"]` % тесту, але дають `{python} numbers["share_big_err"]` % сумарної абсолютної
помилки. У відсотках картина інша: медіанна відносна помилка ≈ 5 % для каменів 0.5–2 ct, але
`{python} numbers["rel_err_small"]` % для дрібних (до 0.5 ct) і
`{python} numbers["rel_err_large"]` % для каменів понад 2 ct. Розрив на графіку біля 1 500 $
походить з даних: у наборі майже немає цін між 1 455 і 1 545 $ (див. лабораторну №2).
Найбільші абсолютні помилки — у дорогих каменів поблизу верхньої межі цін набору (`{python} numbers["n_cap"]` каменів дорожчих за 15 000 $, MAE `{python} usd(numbers["mae_cap"])`):
там ціни обрізані, а інші характеристики (флуоресценція, пропорції, сертифікат, продавець) у дані не
потрапили.

In [ ]:
#| label: save-key
with open("results/key_numbers.json", "w", encoding="utf-8") as f:
    json.dump(numbers, f, ensure_ascii=False, indent=2)

# Обговорення й висновки

**Відповідь на дослідницьке питання.** Чотири характеристики з сертифіката прогнозують ціну нового
діаманта з того самого ринку дуже добре. Обрана модель на тесті помиляється в середньому на
`{python} usd(numbers["test"][1]["MAE, $"])`, тоді як константний прогноз — на
`{python} usd(numbers["test"][0]["MAE, $"])`. Типова (медіанна) відносна помилка становить
`{python} numbers["test"][1]["медіана |помилки| / ціна, %"]` %. Найкраща лінійна модель трохи гірша
(MAE `{python} usd(numbers["test"][-1]["MAE, $"])`), зате пояснює, як саме 4C пов'язані з ціною.

**H1 (вага).** Еластичність `{python} numbers["b1"]`, 95 % ДІ `{python} numbers["b1_lo"]`–`{python} numbers["b1_hi"]`.
Інтервал вузький і цілком лежить вище 1, тож за сталих інших C ціна зростає з вагою швидше, ніж
пропорційно. Дані **узгоджуються з H1**. Оцінка більша за нахил 1.68 з лабораторної №2, бо там вага
не була «очищена» від якості: великі камені частіше мають гіршу чистоту й колір. Застереження:
діагностика показала, що еластичність не стала (стрибки на круглих вагах), а гіпотеза походить з EDA,
тож результат розвідувальний.

**H2 (чистота).** Усі 7 коефіцієнтів чистоти додатні, з вузькими ДІ, і зростають від SI2 до IF
(`{python} "монотонно" if numbers["clar_monotone"] else "не зовсім монотонно"`). Надбавка IF над I1 становить
`{python} numbers["if_prem"]` % (95 % ДІ `{python} numbers["if_lo"]`–`{python} numbers["if_hi"]` %), що більше за
100 %. Дані **узгоджуються з H2** (так само розвідувально).

**Якість прогнозу на нових даних.** Різниця між MAE на перехресній валідації й на тесті мала, отже модель
не перенавчена. Однак тест не повністю незалежний: рішення про логарифми й набір предикторів я ухвалив
після EDA всього набору.

**Межі узагальнення й змішувальні чинники.**

- Дані — один ринок і один рік (2008), лише круглі камені, ціни до 18 823 $. На інші роки, валюти,
  форми каменів чи синтетичні діаманти результат не переноситься.
- Це **спостережні** дані, тому коефіцієнти описують **асоціації** в каталогах, а не причинний вплив.
  Змішувальні чинники, яких немає в даних: продавець і його націнка, лабораторія сертифікації (GIA чи інша),
  флуоресценція, симетрія й полірування, дата пропозиції.
- Ціни взято з каталогу (ціни пропозиції), а не з реальних угод.

**Продовження дослідження.** Зібрати свіжі дані з кількох онлайн-магазинів з датою, продавцем,
лабораторією сертифікації й флуоресценцією. Перевірити зафіксовані моделі (M5 для прогнозу, M4 для
інтерпретації) на них як на справді
незалежному тесті та оцінити, чи збереглися надбавки за «круглі» ваги через 15+ років.

# Бібліографія {.unnumbered}

::: {#refs}
:::